In [0]:
from pyspark.sql.functions import col, sum, count, avg, max, min, year, month, dayofmonth, dayofweek, quarter, date_format, datediff, current_timestamp, lit, when, coalesce
from pyspark.sql.types import IntegerType, DoubleType, DateType, StringType

In [0]:
# fact sales
df_sales = spark.table("electronic_retailer.02_Silver.sales_clean")
df_products = spark.table("electronic_retailer.02_Silver.products_clean")
df_exchange = spark.table("electronic_retailer.02_Silver.exchange_rates_clean")

fact_sales = df_sales.join(
    df_products.select("product_key", "unit_price_usd"),
    "product_key",
    "left"
)

fact_sales = fact_sales.join(
    df_exchange.select(
        col("date").alias("exchange_date"),
        col("currency").alias("exchange_currency"),
        col("exchange").alias("exchange_rate")
    ),
    (fact_sales.order_date == col("exchange_date")) & 
    (fact_sales.currency_code == col("exchange_currency")),
    "left"
)# convert usd

fact_sales = fact_sales \
    .withColumn("revenue_usd", 
                col("quantity") * col("unit_price_usd") * coalesce(col("exchange_rate"), lit(1.0))) \
    .withColumn("delivery_days", 
                when(
                    col("delivery_date").isNotNull() & (datediff(col("delivery_date"), col("order_date")) >= 0),
                    datediff(col("delivery_date"), col("order_date"))
                ).otherwise(None))  

fact_sales = fact_sales.select(
    col("order_number"),
    col("line_item"),
    col("order_date"),
    col("delivery_date"),
    col("customer_key"),
    col("store_key"),
    col("product_key"),
    col("quantity"),
    col("unit_price_usd"),
    col("currency_code"),
    col("exchange_rate"),
    col("revenue_usd"),
    col("delivery_days")
)

fact_sales.write.mode("overwrite").saveAsTable("electronic_retailer.03_Gold.fact_sales")

In [0]:
df_sales = spark.table("electronic_retailer.02_Silver.sales_clean")

dim_date = df_sales.select(col("order_date").alias("date")).distinct() \
    .withColumn("year", year(col("date"))) \
    .withColumn("month", month(col("date"))) \
    .withColumn("day", dayofmonth(col("date"))) \
    .withColumn("quarter", quarter(col("date"))) \
    .withColumn("day_of_week", dayofweek(col("date"))) \
    .withColumn("month_name", date_format(col("date"), "MMMM")) \
    .withColumn("day_name", date_format(col("date"), "EEEE")) \
    .filter(col("date").isNotNull()) \
    .orderBy("date")

dim_date.write.mode("overwrite").saveAsTable("electronic_retailer.03_Gold.dim_date")

In [0]:
df_stores = spark.table("electronic_retailer.02_Silver.stores_clean")

dim_stores = df_stores.select(
    col("store_key"),
    col("country"),
    col("state"),
    col("square_meters"),
    col("open_date")
).withColumn("channel", when(col("store_key") == 0, lit("Online")).otherwise(lit("In-Store")))

dim_stores.write.mode("overwrite").saveAsTable("electronic_retailer.03_Gold.dim_stores")

In [0]:
df_products = spark.table("electronic_retailer.02_Silver.products_clean")

dim_products = df_products.select(
    col("product_key"),
    col("product_name"),
    col("brand"),
    col("color"),
    col("unit_cost_usd"),
    col("unit_price_usd"),
    col("subcategory_key"),
    col("subcategory"),
    col("category_key"),
    col("category")
)
dim_products.write.mode("overwrite").saveAsTable("electronic_retailer.03_Gold.dim_products")

In [0]:
df_customers = spark.table("electronic_retailer.02_Silver.customers_clean")

dim_customers = df_customers.select(
    col("customer_key"),
    col("gender"),
    col("name"),
    col("city"),
    col("state_code"),
    col("state"),
    col("zip_code"),
    col("country"),
    col("continent"),
    col("birthday")
)

dim_customers.write.mode("overwrite").saveAsTable("electronic_retailer.03_Gold.dim_customers")